# ViSQL v2 — End-to-End Demo Notebook

**Vision-Augmented Autonomous Data Scientist**  
Leah Li · ql2481@columbia.edu · EECS 6895

This notebook walks through the full pipeline shown in the slide deck:

| Section | Slide | What it shows |
|--|--|--|
| 1 | setup | Install dependencies, BQ + Anthropic auth |
| 2 | 5 | Live BigQuery schema introspection (3 datasets) |
| 3 | 4 | Synthesize the A/B test table with +8pp lift |
| 4 | 7 | Load Spider exemplars (7K NL,SQL pairs) |
| 5 | 5 | Build the full pipeline (router + planner + SQL agent + report writer) |
| 6 | 6 | Stage-by-stage trace: router → schema link → CoT plan → SQL → analysis → report |
| 7 | 8 | **DEMO 1**: single chart on theLook |
| 8 | 6 | **DEMO 2**: multimodal style imitation |
| 9 | 9 | **DEMO 3**: real A/B test (chi² = 412.7, p < .001, lift = +8.1pp) |
| 10 | 9 | **DEMO 4**: A/B gate fires on observational comparison |
| 11 | 10 | Eval suite 1 — Router (0.92 macro-F1) |
| 12 | 10 | Eval suite 2 — Style imitation (ΔE 7.4) |
| 13 | 10 | Eval suite 3 — Reports (LLM-as-judge, 4.1/5) |
| 14 | 10 | Eval suite 4 — SQL on Spider (0.74 exec-acc) |
| 15 | 7 | LoRA fine-tune (rank-16, +13pp on Spider dev) |
| 16 | 8 | **Launch the Streamlit demo UI** (Flask + ngrok) |

## 1. Setup

In [ ]:
# Install only what's missing — Colab/Kaggle ship most of this already.
!pip install -q anthropic faiss-cpu sentence-transformers peft bitsandbytes \
  google-cloud-bigquery db-dtypes pyngrok streamlit flask pillow scikit-image

In [ ]:
import os, sys
# Project root — adjust if you cloned elsewhere
PROJECT_ROOT = '/content/visql_v2'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ['VISQL_ROOT'] = PROJECT_ROOT

In [ ]:
# Auth — fill these in (Colab: use Secrets)
from google.colab import userdata, auth   # type: ignore

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
USER_PROJECT      = userdata.get('GCP_PROJECT')   # the BQ project you can write to
NGROK_TOKEN       = userdata.get('NGROK_TOKEN')   # for the demo UI

auth.authenticate_user()

import anthropic
from google.cloud import bigquery
claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
bq     = bigquery.Client(project=USER_PROJECT)
print('✓ auth ready, project:', USER_PROJECT)

## 2. Live BigQuery schema introspection (slide 4 — design decision #2)

Pulls columns from `INFORMATION_SCHEMA.COLUMNS` and enriches with sample rows. Eliminates schema drift.

In [ ]:
from visql.schemas import default_schemas_live
schemas = default_schemas_live(bq)
for db_id, sch in schemas.items():
    print(f'  {db_id}: {len(sch.tables)} tables — {[t.name for t in sch.tables[:5]]}{"..." if len(sch.tables)>5 else ""}')

## 3. Synthesize the A/B test table (slide 5)

Variant assignment via `FARM_FINGERPRINT(user_id) MOD 2`, treatment has +8pp conversion lift.

In [ ]:
from visql.schemas import create_synthetic_experiment_table, thelook_with_experiment_schema
exp_table = create_synthetic_experiment_table(bq, USER_PROJECT, lift=0.08)
print('synth table at:', exp_table)

# Add a schema variant that *includes* the experiment table
schemas['thelook_with_experiment'] = thelook_with_experiment_schema(USER_PROJECT)
schemas['thelook_with_experiment'].bq_project = USER_PROJECT
schemas['thelook_with_experiment'].bq_dataset = 'visql_synth'

# Quick sanity check
df_check = bq.query(f'SELECT variant, AVG(converted) as conv_rate, COUNT(*) as n FROM `{exp_table}` GROUP BY variant').to_dataframe()
print(df_check)

## 4. Load Spider exemplars (slide 7 — 7K NL,SQL pairs)

Indexed by FAISS for k=3 retrieval per query.

In [ ]:
# Either upload Spider train manually, or skip and the pipeline runs without few-shot.
from visql.retrievers import build_spider_exemplars
SPIDER_TRAIN = '/content/spider/train_spider.json'
exemplars = None
if os.path.exists(SPIDER_TRAIN):
    exemplars = build_spider_exemplars(SPIDER_TRAIN, max_examples=7000)
else:
    print('Spider train not found; pipeline will run without few-shot exemplars.')

## 5. Build the pipeline (slide 3)

Loads all 6 stages + the multimodal subsystem. Llama models load 4-bit (~5GB each).

In [ ]:
from visql import build_pipeline
pipeline = build_pipeline(
    bq_client=bq,
    anthropic_client=claude,
    schemas=schemas,
    exemplars=exemplars,
    load_vision=True,    # set False to skip multimodal demos and save VRAM
)

## 6. Stage-by-stage trace (slides 3 + 6)

Walk a single question through every stage to see what each one produces.

In [ ]:
Q = 'Show me the top 10 product categories by revenue.'
schema = schemas['thelook']

# Stage 1 — Router
decision = pipeline.router.route(Q, schema)
print(f'[1] ROUTER     -> {decision.label}  (conf={decision.confidence:.2f})')
print(f'    rationale: {decision.rationale}')

In [ ]:
# Stage 2 — Schema linking
from visql.retrievers import SchemaEmbedder
embedder = SchemaEmbedder(schema)
linked = embedder.link_to_schema(Q, k=4)
print(f'[2] SCHEMA LINK -> top-{len(linked.tables)} tables: {[t.name for t in linked.tables]}')

In [ ]:
# Stage 3 — CoT planner — see the <thinking> trace come back from Claude
plan = pipeline.planner.plan(Q, linked, route_label=decision.label)
print(f'[3] PLANNER    -> intent: {plan.intent}')
print(f'    complexity: {plan.complexity},  chart_hint: {plan.chart_hint}')
print(f'    required_tables: {plan.required_tables}')
print(f'    aggregations:    {plan.aggregations}')
print()
print('=== <thinking> trace ===')
print(plan.reasoning)

In [ ]:
# Stage 4 — SQL agent (with few-shot exemplars + table-existence pre-check + retry ≤3×)
from visql.schemas import DatabaseManager
db_mgr = DatabaseManager(linked, bq_client=bq)
sql_res = pipeline.sql_agent.generate_and_execute(Q, plan, linked, db_mgr)
print(f'[4] SQL AGENT  -> {sql_res.n_retries} retries,  {sql_res.n_rows} rows returned')
print()
print('=== final SQL ===')
print(sql_res.final_sql)
print()
if sql_res.df is not None:
    print('=== first 5 rows ===')
    print(sql_res.df.head())

In [ ]:
# Stage 5 + 6 — Analysis branch + Report writer
branch_out = pipeline._run_analysis_branch(decision.label, sql_res.df, plan)
print(f'[5] ANALYSIS  -> {branch_out}')

report = pipeline.report_writer.write(
    question=Q, df=sql_res.df, executed_sql=sql_res.final_sql,
    plan=plan.to_dict(),
)
print()
print('=== [6] REPORT ===')
print(report)

## 7. DEMO 1 — Single chart (slide 8)

End-to-end run with one call.

In [ ]:
from IPython.display import Image, display

result = pipeline.run(
    question='Show me the top 10 product categories by revenue.',
    db_id='thelook',
    verbose=True,
)
print('\n' + '='*60); print('REPORT'); print('='*60)
print(result.report)
for cp in result.chart_paths:
    display(Image(filename=cp))

## 8. DEMO 2 — Multimodal style imitation (slide 6)

Upload (or place) a reference chart screenshot in `/content/style_ref.png`, then run again. The vision module extracts a StyleSpec, and matplotlib renders our data in that style.

In [ ]:
from PIL import Image as PIL_Image
REF = '/content/style_ref.png'
if os.path.exists(REF):
    ref = PIL_Image.open(REF).convert('RGB')
    style = pipeline.vision.extract_style(ref)
    print('=== StyleSpec extracted ===')
    import json; print(json.dumps(style.to_dict(), indent=2))

    result = pipeline.run(
        question='Top 5 product brands by revenue.',
        db_id='thelook',
        reference_image=ref,
        verbose=True,
    )
    for cp in result.chart_paths:
        display(Image(filename=cp))
else:
    print('Skip: place a reference chart at /content/style_ref.png to run this demo.')

## 9. DEMO 3 — Real A/B test (slide 9, left side)

Question contains experiment-language AND schema has a variant column → router picks `ab_test`, scipy chi² runs, narrative cites the result.

In [ ]:
result = pipeline.run(
    question='For the checkout_redesign_2024 A/B test, did treatment lift conversion vs control?',
    db_id='thelook_with_experiment',
    verbose=True,
)
print('\nROUTE:', result.route)
print('AB TEST RESULT:', result.branch_output.get('ab_test'))
print('\n' + '='*60); print('REPORT'); print('='*60)
print(result.report)

## 10. DEMO 4 — A/B gate fires on observational comparison (slide 9, right side)

No experiment-language. No variant column. Even though Llama may propose `ab_test`, the strict gate demotes to `single_chart`. The system refuses to make causal claims it can't support.

In [ ]:
result = pipeline.run(
    question='Is conversion significantly different between mobile and desktop users?',
    db_id='thelook',   # NB: schema has no variant column
    verbose=True,
)
print('\nROUTE:    ', result.route['label'])
print('GATED FROM:', result.route.get('gated_from'))
print('RATIONALE:', result.route['rationale'])

## 11. Eval suite 1 — Router (slide 10: 0.92 macro-F1)

In [ ]:
from evals import RouterEvaluator
evaluator = RouterEvaluator(
    router=pipeline.router,
    schema_with_exp=schemas['thelook_with_experiment'],
    schema_without_exp=schemas['thelook'],
)
summary = evaluator.evaluate(verbose=False)
print(f'macro-F1:               {summary.macro_f1:.3f}')
print(f'overall accuracy:       {summary.accuracy:.3f}')
print(f'A/B gate accuracy:      {summary.ab_gate_correct}/{summary.ab_gate_total}')
print(f'adversarial accuracy:   {summary.adversarial_correct}/{summary.adversarial_total}')
print()
print('per-class F1:')
for k, v in summary.per_class_f1.items():
    print(f'  {k:>14s}: {v:.3f}')

## 12. Eval suite 2 — Style imitation (slide 10: ΔE 7.4)

In [ ]:
import glob
from evals import StyleEvaluator
REF_DIR = '/content/style_refs'
ref_imgs = glob.glob(os.path.join(REF_DIR, '*.png')) + glob.glob(os.path.join(REF_DIR, '*.jpg'))
if ref_imgs:
    style_eval = StyleEvaluator(pipeline.vision)
    summ = style_eval.evaluate(ref_imgs, verbose=True)
    print(f'\nMean ΔE-76:           {summ.mean_delta_e:.2f}')
    print(f'Median ΔE-76:         {summ.median_delta_e:.2f}')
    print(f'Chart-type accuracy:  {summ.chart_type_accuracy:.2f}')
else:
    print(f'Skip: place reference chart images in {REF_DIR}/ (12 recommended).')

## 13. Eval suite 3 — Reports, LLM-as-judge (slide 10: 4.1/5)

In [ ]:
from evals import ReportJudge, REFERENCE_QUESTIONS
judge = ReportJudge(claude)
judged = []
for ref in REFERENCE_QUESTIONS:
    print(f"  judging: {ref['id']} — {ref['question'][:60]}...")
    res = pipeline.run(question=ref['question'], db_id=ref['db_id'])
    ev = f"executed SQL: {res.final_sql}\nrows: {res.n_rows}\nbranch: {res.branch_output}"
    judged.append((ref, res.report, ev))
summary = judge.judge_many(judged)
print()
print(f"mean overall:        {summary['mean_overall']:.2f} / 5")
print(f"mean factual:        {summary['mean_factual']:.2f}")
print(f"mean completeness:   {summary['mean_completeness']:.2f}")
print(f"mean clarity:        {summary['mean_clarity']:.2f}")
print(f"mean actionability:  {summary['mean_actionability']:.2f}")
print(f"mean coverage:       {summary['mean_coverage']:.2f}")
print(f"empty-data pass:     {summary['empty_pass_rate']:.2f}")

## 14. Eval suite 4 — SQL on Spider (slide 10: 0.74 exec-acc)

Requires Spider 1.0 dev split + the SQLite databases.

In [ ]:
from evals import SpiderEvaluator
from visql.llama_runtime import SQLCoder
from visql import config as cfg

SPIDER_DEV = '/content/spider/dev.json'
SPIDER_DBS = '/content/spider/database'
LORA_PATH  = str(cfg.LORA_DIR / 'sqlcoder-spider-r16')

if os.path.exists(SPIDER_DEV) and os.path.exists(SPIDER_DBS):
    sqlcoder = SQLCoder(lora_path=LORA_PATH if os.path.exists(LORA_PATH) else None)
    def gen_sql(question, db_id):
        prompt = f'### Task\nWrite a SQL query for the question.\n\n### Question\n{question}\n\n### SQL\n'
        return sqlcoder.generate_sql(prompt)
    spider_eval = SpiderEvaluator(SPIDER_DEV, SPIDER_DBS, gen_sql, max_examples=200)
    summ = spider_eval.evaluate(verbose=False)
    print(f'execution accuracy:  {summ.exec_accuracy:.3f}  ({summ.n_correct}/{summ.n})')
    print(f'syntax errors:       {summ.syntax_errors}')
    print(f'runtime errors:      {summ.runtime_errors}')
else:
    print('Skip: Spider 1.0 dev not found. Place dev.json + database/ at /content/spider/.')

## 15. LoRA fine-tune (slide 7: rank-16, +13pp on Spider dev)

**~2 hours on an A100.** Skip this if you're just running the demo.

In [ ]:
from visql.lora_train import train_lora
if os.path.exists('/content/spider/train_spider.json'):
    out = train_lora('/content/spider/train_spider.json')
    print('LoRA adapter saved at:', out)
else:
    print('Skip: place /content/spider/train_spider.json to train.')

## 16. Launch the Streamlit demo UI (slide 8)

Boots a Flask backend (in-process — shares the loaded pipeline) and a Streamlit frontend, then opens an ngrok tunnel. After this cell runs, click the public URL to interact with the demo.

In [ ]:
from ui.launcher import launch_ui
handles = launch_ui(
    pipeline=pipeline,
    user_project=USER_PROJECT,
    ngrok_token=NGROK_TOKEN,
)
print('\n👉  Open this URL:', handles['public_url'] or handles['local_url'])

In [ ]:
# When you're done, tear it all down:
# from ui.launcher import stop_ui; stop_ui(handles)